# Module 5: FedAvg Baselines

This focused notebook owns the clean FedAvg and attacked FedAvg handoff artifacts for Module 5.


## Stage Goal

Run the same FedAvg server twice: once with no malicious clients and once with the configured Module 4 malicious-client recipe. Save both result JSON files, update diagnostics, and an inline update-norm plot.


## 1. Notebook Setup

Load the split-notebook helper layer and keep imports independent of whether Jupyter starts from the repository root or from `5_Defensive_FL/`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
MODULE_DIR = cwd if cwd.name == "5_Defensive_FL" else cwd / "5_Defensive_FL"
MODULE4_DIR = MODULE_DIR.parent / "4_Adversarial_FL"
MODULE4_SRC_DIR = MODULE4_DIR / "src"
REPO_ROOT = MODULE_DIR.parent

for path in (REPO_ROOT, MODULE4_DIR, MODULE4_SRC_DIR, MODULE_DIR):
    path_str = str(path.resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from src.notebook_utils import (
    prepare_context,
    record_config_snapshot,
    run_fedavg_baselines,
    save_update_diagnostics,
    run_result_path,
    validate_artifacts,
    validate_result_collection,
)


## 2. Configuration

Edit this visible `CONFIG` cell to change data, FL, attack, handoff, or artifact settings. No YAML file is required for the focused notebook path.


In [ ]:
CONFIG = {'data_config': {'dataset_path': './Data/Imagenette',
                 'dataset_name': 'Imagenette',
                 'non_iid_per': 0,
                 'eval_subset': 'attack_eval',
                 'validation_split': {'enabled': True, 'seed': 42, 'selection_fraction': 0.5}},
 'global_config': {'seed': 42, 'device': 'cuda'},
 'module4_handoff': {'enabled': True,
                     'artifacts_dir': '../4_Adversarial_FL/artifacts',
                     'target_checkpoint': 'module4_v3_target.pt',
                     'surrogate_checkpoint': 'module4_surrogate.pt'},
 'artifacts': {'dir': 'artifacts',
               'config_snapshot': 'module5_baselines_config_used.json',
               'clean_baseline_metrics': 'module5_clean_fedavg.json',
               'attacked_baseline_metrics': 'module5_attacked_fedavg.json',
               'update_diagnostics': 'module5_update_diagnostics.json',
               'update_norm_plot': 'module5_update_norms.png'},
 'stage': {'name': 'fedavg_baselines', 'notebook': 'fedavg_baselines.ipynb'},
 'model_config': {'module': 'model',
                  'name': 'MobileNetV3Transfer',
                  'kwargs': {'pretrained': False, 'num_classes': 10, 'dropout': 0.1}},
 'algorithms': {'FedAvg': {'fed_config': {'algorithm': 'FedAvg',
                                          'fraction_clients': 0.2,
                                          'num_clients': 50,
                                          'num_rounds': 12,
                                          'num_epochs': 3,
                                          'batch_size': 64,
                                          'global_stepsize': 1.0,
                                          'local_stepsize': 0.002,
                                          'criterion': 'torch.nn.CrossEntropyLoss'},
                           'optim_config': {}}},
 'defense': {'name': 'fedavg'},
 'attack': {'seed': 42,
            'malicious_fraction': 0.1,
            'malicious_client_selection': {'mode': 'seeded_random', 'client_ids': []},
            'start_round': 3,
            'attack': {'type': 'pgd',
                       'poison_rate': 0.2,
                       'target_label': 0,
                       'epsilon': 0.03137255,
                       'step_size': 0.00784314,
                       'iters': 20,
                       'criterion': 'torch.nn.CrossEntropyLoss',
                       'poison_rate_schedule': {'type': 'linear', 'start': 0.05, 'end': 0.2}},
            'surrogate': {'checkpoint': '../4_Adversarial_FL/artifacts/module4_surrogate.pt',
                          'checkpoint_source': 'train_surrogate.ipynb',
                          'pretrained': False,
                          'finetune_epochs': 0,
                          'local_finetune_epochs': 0,
                          'learning_rate': 0.001,
                          'weight_decay': 0.0,
                          'batch_size': 64,
                          'freeze_backbone': False,
                          'early_stop_patience': 0}}}

context = prepare_context(CONFIG, stage_name="fedavg_baselines")
config_snapshot_path = record_config_snapshot(context)
config_snapshot_path


## 3. Clean and Attacked FedAvg

The clean run sets malicious fraction and poison rate to zero. The attacked run uses the configured PGD malicious-client recipe.


In [ ]:
run_results = run_fedavg_baselines(context)
baseline_validation = validate_result_collection(
    context,
    run_results,
    required_runs=["clean_fedavg", "attacked_fedavg"],
)
validate_artifacts(run_result_path(context, name) for name in run_results)
display(pd.DataFrame(baseline_validation))


## 4. Update Diagnostics

Use the attacked FedAvg run to record client update norms, malicious-client flags, cosine-to-mean behavior, and distance to the coordinate-wise median.


In [ ]:
norm_rows = save_update_diagnostics(context, run_results["attacked_fedavg"])
validate_artifacts([
    context.artifact_path("module5_update_diagnostics.json"),
    context.artifact_path("module5_update_norms.png"),
])
display(pd.DataFrame(norm_rows).head())
display(Image(filename=str(context.artifact_path("module5_update_norms.png"))))


## Handoff Artifacts

Run one or more `*_defense.ipynb` notebooks after these files exist.


In [ ]:
handoff_artifacts = validate_artifacts([
    config_snapshot_path,
    context.artifact_path("module5_clean_fedavg.json"),
    context.artifact_path("module5_attacked_fedavg.json"),
    context.artifact_path("module5_update_diagnostics.json"),
    context.artifact_path("module5_update_norms.png"),
])
handoff_artifacts
